# Train defect detection (YOLOv8-nano) on Colab

Run this notebook **top to bottom**. It will:
1. Install dependencies
2. Load NEU-DET (upload `archive-2.zip` or use Google Drive)
3. Convert to YOLO format and train YOLOv8-nano
4. Export TFLite for Raspberry Pi

**Two ways to get the code:**
- **Upload files:** Upload this notebook + `convert_neu_det_to_yolo.py` + `train_yolo.py` + `archive-2.zip` to Colab (no clone).
- **Clone repo:** Upload only `archive-2.zip`, then the notebook will clone the repo to get the scripts.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics Pillow
print('Done.')

## 2. Get the dataset (choose one)

**Option A:** Upload `archive-2.zip` when prompted.

**Option B:** If the zip is in your Google Drive, mount Drive and set `ZIP_PATH` below.

In [ ]:
# Option A: Upload the zip (run this cell and use the file picker)
from google.colab import files
uploaded = files.upload()  # pick archive-2.zip
ZIP_PATH = 'archive-2.zip'  # name after upload

In [ ]:
# Option B (optional): Use zip from Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/archive-2.zip'  # your path

## 3. Get scripts and unzip dataset

If you **uploaded** the Python scripts, we use them from `/content`. Otherwise we clone the repo. Then unzip NEU-DET.

In [ ]:
import os

# Use uploaded scripts in /content, or clone the repo
SCRIPT_NAME = 'convert_neu_det_to_yolo.py'
if os.path.isfile(f'/content/{SCRIPT_NAME}'):
    WORK_DIR = '/content'
    print('Using uploaded scripts in /content')
else:
    WORK_DIR = '/content/Capstone_Joleen'
    if not os.path.isdir(WORK_DIR):
        !git clone https://github.com/kneeklers/Capstone_Joleen.git "{WORK_DIR}"
    print('Using scripts from cloned repo')
os.chdir(WORK_DIR)
print('Working in', os.getcwd())

# Unzip dataset (ZIP_PATH from cell above: upload or Drive)
!unzip -q -o "{ZIP_PATH}" -d /content
assert os.path.isdir('/content/NEU-DET'), 'Expected /content/NEU-DET after unzip. Check ZIP_PATH.'
print('NEU-DET extracted to /content/NEU-DET')

## 4. Convert NEU-DET to YOLO format

In [ ]:
!python convert_neu_det_to_yolo.py --input-dir /content/NEU-DET --output-dir /content/yolo_dataset --val-ratio 0.2

## 5. Train YOLOv8-nano and export TFLite

In [ ]:
!python train_yolo.py --data-dir /content/yolo_dataset --output-dir /content/output --epochs 80 --imgsz 640 --batch 16

## 6. Download results for the Pi

Run the cell below to download a zip with the TFLite model and `labels.txt`.

In [ ]:
import glob
from google.colab import files

os.chdir('/content/output')
tflite = glob.glob('*.tflite')
labels = 'labels.txt'
if tflite:
    !zip -j defect_model_for_pi.zip {' '.join(tflite)} {labels}
    files.download('defect_model_for_pi.zip')
    print('Downloaded defect_model_for_pi.zip')
else:
    print('No .tflite in /content/output. Check training logs above.')